# NIFTY 50 Cross-Sectional Momentum + Mean Reversion
## Research Notebook

This notebook walks through every layer of the strategy interactively:

| # | Section |
|---|---|
| 1 | Environment setup & imports |
| 2 | Data download and quality checks |
| 3 | Signal construction and visualisation |
| 4 | Backtest execution |
| 5 | Performance analytics |
| 6 | Output generation |
| 7 | Parameter sensitivity analysis |
| 8 | Walk-forward validation |

> **Execution note** — Run cells top-to-bottom. The first run downloads data from Yahoo Finance and caches it as Parquet. Subsequent runs are fast.

---
## 1. Environment Setup

In [1]:
import sys, warnings
from pathlib import Path

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use('inline')          # Jupyter inline rendering
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Point Python at the project root so strategy.* is importable ────────
PROJECT_ROOT = Path().resolve().parent   # notebooks/ → project root
sys.path.insert(0, str(PROJECT_ROOT))

from strategy.signals  import (
    calculate_momentum, rank_momentum,
    get_long_candidates, get_short_candidates,
    calculate_rsi, get_entry_signals,
    calculate_volatility_filter, nifty_regime_filter,
)
from strategy.portfolio import (
    apply_regime_filter, apply_volatility_filter, cap_positions,
    sector_neutral_selection,
)
from strategy.backtest import BacktestEngine, BacktestConfig
from strategy.metrics  import (
    calculate_cagr, calculate_sharpe, calculate_sortino,
    calculate_max_drawdown, calculate_calmar,
    calculate_win_rate, calculate_profit_factor,
    calculate_monthly_returns, generate_performance_summary,
    plot_equity_curve, plot_drawdown,
    plot_monthly_heatmap, plot_stock_contribution,
)
from main import (
    NIFTY50_TICKERS, NIFTY50_SECTORS, NIFTY_INDEX_TICKER,
    download_data, download_nifty_index, build_signals,
)

print('Setup complete. Project root:', PROJECT_ROOT)

Setup complete. Project root: /Users/anirudh/Documents/Quant/MoRevStrategy


---
## 2. Data Download & Quality Checks

In [2]:
# ── Backtest date range ──────────────────────────────────────────────────
START_DATE = '2019-01-01'
END_DATE   = '2024-12-31'

DATA_DIR   = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Download (or load from cache) OHLCV data
price_data  = download_data(NIFTY50_TICKERS, START_DATE, END_DATE, DATA_DIR)
nifty_close = download_nifty_index(START_DATE, END_DATE, DATA_DIR)

close  = price_data['close']
open_  = price_data['open']
high   = price_data['high']
low    = price_data['low']

# Align on common trading days
common_idx = close.index.intersection(open_.index).intersection(nifty_close.index)
close, open_, high, low = (df.loc[common_idx] for df in [close, open_, high, low])
nifty_close = nifty_close.loc[common_idx]

# Drop tickers with >20% missing data, then forward-fill
min_obs = int(len(common_idx) * 0.80)
close   = close.dropna(thresh=min_obs, axis=1)
for df in [open_, high, low]:
    df.drop(columns=[c for c in df.columns if c not in close.columns],
            errors='ignore', inplace=True)
close  = close.ffill()
open_  = open_.reindex(columns=close.columns).ffill()
high   = high.reindex(columns=close.columns).ffill()
low    = low.reindex(columns=close.columns).ffill()

print(f'Universe  : {len(close.columns)} stocks')
print(f'Period    : {close.index[0].date()}  →  {close.index[-1].date()}')
print(f'Bar count : {len(close):,} trading days')
close.tail(3)

[DATA] Loading cached data  → /Users/anirudh/Documents/Quant/MoRevStrategy/data/stocks_2019-01-01_2024-12-31.pkl
Universe  : 48 stocks
Period    : 2019-01-02  →  2024-12-30
Bar count : 1,477 trading days


,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJAJFINSV.NS,BAJFINANCE.NS,BHARTIARTL.NS,BPCL.NS,BRITANNIA.NS,CIPLA.NS,COALINDIA.NS,DIVISLAB.NS,DRREDDY.NS,...,NTPC.NS,ONGC.NS,POWERGRID.NS,RELIANCE.NS,SBILIFE.NS,SBIN.NS,SUNPHARMA.NS,TATACONSUM.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,ULTRACEMCO.NS,UPL.NS,WIPRO.NS
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2024-12-26,2399.0608,1237.9272,7232.0239,2238.2241,1075.7805,8659.0459,1558.5408,677.3951,1586.2393,272.0765,4700.0557,1474.7958,355.9522,5860.0669,1346.6096,...,323.5217,223.4654,297.0059,1211.7106,1404.4210,782.3555,1812.5399,894.3218,137.1613,3957.0210,1650.9330,3313.0793,11388.4619,495.1190,285.9540
2024-12-27,2408.7559,1224.7413,7217.2334,2247.1279,1076.5299,8707.6143,1578.5310,686.5388,1586.5863,270.1896,4707.6064,1490.6760,350.3359,5854.3926,1380.6934,...,323.8601,220.3494,296.1444,1216.1927,1400.6833,770.0297,1832.1287,901.2703,135.7444,3952.9871,1663.5189,3299.2710,11334.7021,497.4973,289.7505
2024-12-30,2591.0657,1212.6005,7284.9111,2263.7981,1069.0364,8562.8828,1575.9321,684.6006,1573.7437,269.9595,4737.3667,1505.3197,347.9420,6017.5996,1368.2225,...,320.3798,216.3963,294.3737,1205.8838,1399.2380,759.1000,1854.4243,896.8530,133.7606,3947.2446,1691.8978,3247.5767,11217.2471,496.2091,284.6886


In [4]:
# Missing-data audit
missing_pct = (close.isna().sum() / len(close) * 100).sort_values(ascending=False)
print('Tickers with any missing data (after ffill):',
      (missing_pct > 0).sum())
missing_pct[missing_pct > 0].head(10)

Tickers with any missing data (after ffill): 0


Series([], dtype: float64)

In [5]:
# Visualise normalised price evolution of all universe stocks
fig, ax = plt.subplots(figsize=(15, 6))
normed = close.div(close.iloc[0])          # rebase to 1.0 on first bar
for col in normed.columns:
    ax.plot(normed[col], alpha=0.25, linewidth=0.6, color='steelblue')

# Overlay NIFTY 50
nifty_norm = nifty_close / nifty_close.iloc[0]
ax.plot(nifty_norm, color='darkorange', linewidth=2.0, label='NIFTY 50')
ax.set_title('NIFTY 50 Universe — Normalised Price (Base = 1.0)', fontsize=13)
ax.set_ylabel('Normalised Price')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Signal Construction & Visualisation

In [6]:
# ── 3.1 Cross-sectional 20-day momentum ─────────────────────────────────
momentum = calculate_momentum(close, window=20)
ranks    = rank_momentum(momentum)

# Snapshot: most recent momentum scores
latest_mom = momentum.iloc[-1].sort_values(ascending=False)
print('Top 10 momentum stocks (latest bar):')
print(latest_mom.head(10).to_string())
print('\nBottom 10 momentum stocks (latest bar):')
print(latest_mom.tail(10).to_string())

Top 10 momentum stocks (latest bar):
DRREDDY.NS      0.1452
APOLLOHOSP.NS   0.0711
SUNPHARMA.NS    0.0578
ADANIENT.NS     0.0525
WIPRO.NS        0.0511
BAJFINANCE.NS   0.0475
HCLTECH.NS      0.0438
INFY.NS         0.0259
ADANIPORTS.NS   0.0239
TECHM.NS        0.0167

Bottom 10 momentum stocks (latest bar):
RELIANCE.NS     -0.0631
POWERGRID.NS    -0.0663
JSWSTEEL.NS     -0.0694
ASIANPAINT.NS   -0.0772
UPL.NS          -0.0812
HINDALCO.NS     -0.0840
NTPC.NS         -0.0887
COALINDIA.NS    -0.0925
ONGC.NS         -0.0937
HEROMOTOCO.NS   -0.1206


In [7]:
# Momentum rank distribution over time for two illustrative stocks
sample_tickers = close.columns[:4].tolist()

fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
for ax, ticker in zip(axes.flat, sample_tickers):
    ax.plot(ranks[ticker], color='royalblue', linewidth=0.8)
    ax.axhline(10,  color='green',  linestyle='--', linewidth=1.0, label='Long threshold')
    ax.axhline(len(close.columns) - 10, color='red', linestyle='--',
               linewidth=1.0, label='Short threshold')
    ax.set_title(f'{ticker} — Momentum Rank (1 = best)', fontsize=11)
    ax.set_ylabel('Rank')
    ax.invert_yaxis()     # rank 1 at top
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Momentum Rank Over Time (selected stocks)', fontsize=13)
plt.tight_layout()
plt.show()

In [8]:
# ── 3.2 RSI(2) — short-term mean reversion timer ────────────────────────
rsi = calculate_rsi(close, period=2)

ticker_demo = close.columns[0]    # pick first ticker for illustration

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True)

ax1.plot(close[ticker_demo], color='navy', linewidth=1.0)
ax1.set_title(f'{ticker_demo} — Close Price', fontsize=11)
ax1.set_ylabel('Price (₹)')
ax1.grid(True, alpha=0.3)

ax2.plot(rsi[ticker_demo], color='purple', linewidth=0.9)
ax2.axhline(10,  color='green', linestyle='--', linewidth=1.2, label='RSI < 10 → Long entry')
ax2.axhline(90,  color='red',   linestyle='--', linewidth=1.2, label='RSI > 90 → Short entry')
ax2.axhline(50,  color='gray',  linestyle=':',  linewidth=0.8, label='RSI = 50 → Exit')
ax2.fill_between(rsi.index, rsi[ticker_demo], 10,
                 where=(rsi[ticker_demo] < 10), color='green', alpha=0.25, label='Oversold zone')
ax2.fill_between(rsi.index, rsi[ticker_demo], 90,
                 where=(rsi[ticker_demo] > 90), color='red',   alpha=0.25, label='Overbought zone')
ax2.set_ylim(0, 100)
ax2.set_title(f'{ticker_demo} — RSI(2)', fontsize=11)
ax2.set_ylabel('RSI')
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [9]:
# ── 3.3 NIFTY 50 regime filter ──────────────────────────────────────────
regime = nifty_regime_filter(nifty_close, ma_window=50)
ma50   = nifty_close.rolling(50).mean()

bull_days = (regime == 1).sum()
bear_days = (regime == -1).sum()
print(f'Bull regime days : {bull_days:,}  ({bull_days / len(regime) * 100:.1f}%)')
print(f'Bear regime days : {bear_days:,}  ({bear_days / len(regime) * 100:.1f}%)')

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(nifty_close, color='navy', linewidth=1.2, label='NIFTY 50')
ax.plot(ma50, color='orange', linewidth=1.5, label='50-day MA')
ax.fill_between(nifty_close.index, nifty_close, ma50,
                where=(regime == 1),  alpha=0.15, color='green', label='Bull regime')
ax.fill_between(nifty_close.index, nifty_close, ma50,
                where=(regime == -1), alpha=0.15, color='red',   label='Bear regime')
ax.set_title('NIFTY 50 Regime Filter (50-day MA)', fontsize=13)
ax.set_ylabel('Index Level')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Bull regime days : 966  (65.4%)
Bear regime days : 511  (34.6%)


In [10]:
# ── 3.4 Combined entry signals ───────────────────────────────────────────
cfg = BacktestConfig()
long_e, short_e, rsi_full = build_signals(
    close, nifty_close, cfg,
    use_sector_neutrality=False,
    sector_map=NIFTY50_SECTORS,
)

daily_long_count  = long_e.sum(axis=1)
daily_short_count = short_e.sum(axis=1)

print(f'Days with at least 1 LONG  signal : {(daily_long_count  > 0).sum():,}')
print(f'Days with at least 1 SHORT signal : {(daily_short_count > 0).sum():,}')

fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(daily_long_count.index,  daily_long_count,  color='green', alpha=0.6, label='Long entries')
ax.bar(daily_short_count.index, -daily_short_count, color='red',  alpha=0.6, label='Short entries')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Daily Entry Signal Count (Long = above zero, Short = below zero)', fontsize=12)
ax.set_ylabel('Number of positions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Days with at least 1 LONG  signal : 334
Days with at least 1 SHORT signal : 0


---
## 4. Backtest Execution

In [11]:
print('Running backtest engine …')
engine  = BacktestEngine(cfg)
results = engine.run(
    close, open_, high, low,
    long_e, short_e, rsi_full,
    nifty_close,
)

equity_curve  = results['equity_curve']
daily_returns = results['daily_returns']
trade_log     = results['trade_log']

print(f'Backtest complete.  Trades : {len(trade_log):,}')

Running backtest engine …
Backtest complete.  Trades : 423


---
## 5. Performance Analytics

In [12]:
summary = generate_performance_summary(
    equity_curve, daily_returns, trade_log, cfg.initial_capital
)
print(summary.to_string(index=False))

                     Metric         Value
        Initial Capital (₹) ₹   1,000,000
          Final Capital (₹) ₹   1,307,785
           Total Return (%)        30.78%
                   CAGR (%)         4.68%
               Sharpe Ratio        -0.280
              Sortino Ratio        -0.424
               Calmar Ratio         0.484
           Max Drawdown (%)        -9.68%
               Win Rate (%)        64.30%
              Profit Factor         1.325
               Total Trades           423
                Long Trades           423
               Short Trades             0
  Avg Holding Period (days)           3.9
Total Transaction Costs (₹) ₹     186,246
               Exits — TIME            64
          Exits — STOP_LOSS            75
           Exits — RSI_EXIT           284


In [13]:
# Equity curve vs NIFTY 50
nifty_bm = nifty_close.reindex(equity_curve.index, method='ffill')
fig = plot_equity_curve(equity_curve, benchmark=nifty_bm)
fig.savefig(OUTPUT_DIR / 'equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
# Drawdown chart
fig = plot_drawdown(equity_curve)
fig.savefig(OUTPUT_DIR / 'drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [15]:
# Monthly returns heatmap
fig = plot_monthly_heatmap(equity_curve)
if fig:
    fig.savefig(OUTPUT_DIR / 'monthly_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [16]:
# Per-stock PnL contribution
fig = plot_stock_contribution(trade_log)
if fig:
    fig.savefig(OUTPUT_DIR / 'stock_contribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [17]:
# Trade log deep dive
print('=== Trade Log Preview ===')
print(trade_log.head(10).to_string())
print(f'\n=== Exit Reason Breakdown ===')
print(trade_log['exit_reason'].value_counts())
print(f'\n=== PnL by Direction ===')
print(trade_log.groupby('direction')['net_pnl'].describe())

=== Trade Log Preview ===
         ticker direction entry_date  exit_date  entry_price  exit_price    shares    notional  gross_pnl  entry_cost  exit_cost    net_pnl  pnl_pct  holding_days exit_reason
0   RELIANCE.NS      LONG 2019-11-06 2019-11-07     645.3963    652.5093  221.3479 142857.1400  1574.4400    183.7500   180.0400  1210.6600   0.8470             1    RSI_EXIT
1    SBILIFE.NS      LONG 2019-11-06 2019-11-13     962.7077    976.2426  148.3910 142857.1400  2008.4500    183.7500   180.5300  1644.1700   1.1510             7    RSI_EXIT
2  EICHERMOT.NS      LONG 2019-11-06 2019-11-14    2021.4555   2039.2529   70.6704 142857.1400  1257.7500    183.7500   179.6800   894.3300   0.6260             8        TIME
3   RELIANCE.NS      LONG 2019-11-07 2019-11-13     641.9517    658.6379  223.1950 143280.4300  3724.2800    184.2900   182.9600  3357.0200   2.3430             6    RSI_EXIT
4  TATASTEEL.NS      LONG 2019-11-11 2019-11-18      33.1875     34.4084 4301.6503 142760.9100  525

In [18]:
# Return distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Daily returns histogram
dr_clean = daily_returns.dropna()
axes[0].hist(dr_clean * 100, bins=80, color='steelblue', edgecolor='none', alpha=0.75)
axes[0].axvline(dr_clean.mean() * 100, color='red',    linewidth=1.5, label=f'Mean: {dr_clean.mean()*100:.3f}%')
axes[0].axvline(0,                      color='black',  linewidth=0.8, linestyle='--')
axes[0].set_title('Daily Return Distribution', fontsize=12)
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-trade PnL % histogram
if not trade_log.empty:
    axes[1].hist(trade_log['pnl_pct'], bins=50, color='coral', edgecolor='none', alpha=0.75)
    axes[1].axvline(trade_log['pnl_pct'].mean(), color='darkred', linewidth=1.5,
                    label=f"Mean: {trade_log['pnl_pct'].mean():.2f}%")
    axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1].set_title('Per-Trade PnL % Distribution', fontsize=12)
    axes[1].set_xlabel('Trade PnL (%)')
    axes[1].set_ylabel('Count')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [19]:
# Rolling 6-month Sharpe ratio
window = 126  # ~6 months
rfr_d  = 0.065 / 252
roll_sharpe = (
    (daily_returns - rfr_d)
    .rolling(window).mean()
    .div(daily_returns.rolling(window).std())
    * np.sqrt(252)
)

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(roll_sharpe, color='darkorchid', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(1, color='green', linewidth=0.8, linestyle=':', label='Sharpe = 1')
ax.fill_between(roll_sharpe.index, roll_sharpe, 0,
                where=(roll_sharpe > 0), color='green', alpha=0.15)
ax.fill_between(roll_sharpe.index, roll_sharpe, 0,
                where=(roll_sharpe < 0), color='red',   alpha=0.15)
ax.set_title('Rolling 6-Month Sharpe Ratio', fontsize=12)
ax.set_ylabel('Sharpe Ratio')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Parameter Sensitivity Analysis

Sweep two key parameters — momentum window and RSI entry threshold — to understand robustness.

In [20]:
mom_windows   = [10, 15, 20, 25, 30]
rsi_thresholds = [5, 10, 15, 20]

sensitivity_records = []

for mw in mom_windows:
    for rsi_t in rsi_thresholds:
        try:
            cfg_s = BacktestConfig(
                momentum_window=mw,
                rsi_long_entry=float(rsi_t),
                rsi_short_entry=100.0 - rsi_t,
            )
            le_s, se_s, rsi_s = build_signals(close, nifty_close, cfg_s)
            eng_s = BacktestEngine(cfg_s)
            res_s = eng_s.run(close, open_, high, low, le_s, se_s, rsi_s)

            cagr_s   = calculate_cagr(res_s['equity_curve'])
            sharpe_s = calculate_sharpe(res_s['daily_returns'])
            mdd_s, _ = calculate_max_drawdown(res_s['equity_curve'])
            n_trades = len(res_s['trade_log'])

            sensitivity_records.append({
                'momentum_window': mw,
                'rsi_threshold':   rsi_t,
                'cagr_pct':        round(cagr_s   * 100, 2),
                'sharpe':          round(sharpe_s,       3),
                'max_dd_pct':      round(mdd_s    * 100, 2),
                'n_trades':        n_trades,
            })
            print(f'  mw={mw:2d}  rsi_t={rsi_t:2d}  '
                  f'CAGR={cagr_s*100:.1f}%  Sharpe={sharpe_s:.2f}  MDD={mdd_s*100:.1f}%')
        except Exception as exc:
            print(f'  mw={mw}  rsi_t={rsi_t}  FAILED: {exc}')

sens_df = pd.DataFrame(sensitivity_records)
print('\nSensitivity grid complete.')

  mw=10  rsi_t= 5  CAGR=0.3%  Sharpe=-9.65  MDD=-1.1%
  mw=10  rsi_t=10  CAGR=0.5%  Sharpe=-3.34  MDD=-2.5%
  mw=10  rsi_t=15  CAGR=-0.1%  Sharpe=-2.39  MDD=-6.2%
  mw=10  rsi_t=20  CAGR=0.5%  Sharpe=-1.58  MDD=-6.2%
  mw=15  rsi_t= 5  CAGR=1.1%  Sharpe=-3.24  MDD=-1.5%
  mw=15  rsi_t=10  CAGR=1.6%  Sharpe=-1.79  MDD=-2.7%
  mw=15  rsi_t=15  CAGR=1.3%  Sharpe=-1.38  MDD=-7.7%
  mw=15  rsi_t=20  CAGR=1.6%  Sharpe=-1.00  MDD=-7.8%
  mw=20  rsi_t= 5  CAGR=1.1%  Sharpe=-2.49  MDD=-2.8%
  mw=20  rsi_t=10  CAGR=1.6%  Sharpe=-1.29  MDD=-6.3%
  mw=20  rsi_t=15  CAGR=2.3%  Sharpe=-0.84  MDD=-8.5%
  mw=20  rsi_t=20  CAGR=3.0%  Sharpe=-0.56  MDD=-8.7%
  mw=25  rsi_t= 5  CAGR=1.4%  Sharpe=-2.11  MDD=-5.2%
  mw=25  rsi_t=10  CAGR=2.7%  Sharpe=-0.99  MDD=-6.6%
  mw=25  rsi_t=15  CAGR=3.9%  Sharpe=-0.49  MDD=-13.9%
  mw=25  rsi_t=20  CAGR=4.7%  Sharpe=-0.28  MDD=-9.7%
  mw=30  rsi_t= 5  CAGR=1.4%  Sharpe=-1.98  MDD=-4.0%
  mw=30  rsi_t=10  CAGR=1.6%  Sharpe=-1.16  MDD=-5.8%
  mw=30  rsi_t=15  CAGR=0.

In [21]:
# Sharpe ratio heatmap across the parameter grid
if not sens_df.empty:
    pivot_sharpe = sens_df.pivot(
        index='momentum_window', columns='rsi_threshold', values='sharpe'
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    vabs = max(pivot_sharpe.abs().values.max(), 0.1)
    norm = mcolors.TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs)
    im   = ax.imshow(pivot_sharpe.values, cmap='RdYlGn', norm=norm, aspect='auto')
    ax.set_xticks(range(len(pivot_sharpe.columns)))
    ax.set_xticklabels([f'RSI<{c}' for c in pivot_sharpe.columns])
    ax.set_yticks(range(len(pivot_sharpe.index)))
    ax.set_yticklabels([f'MomW={r}' for r in pivot_sharpe.index])
    for r in range(len(pivot_sharpe.index)):
        for c in range(len(pivot_sharpe.columns)):
            ax.text(c, r, f"{pivot_sharpe.values[r,c]:.2f}",
                    ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax, label='Annualised Sharpe')
    ax.set_title('Parameter Sensitivity — Sharpe Ratio', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 8. Walk-Forward Validation

12-month training → 3-month out-of-sample test, rolling forward.

In [22]:
from main import walk_forward_test

param_grid = [
    {'momentum_window': 15, 'rsi_long_entry': 10.0, 'rsi_short_entry': 90.0},
    {'momentum_window': 20, 'rsi_long_entry': 10.0, 'rsi_short_entry': 90.0},
    {'momentum_window': 20, 'rsi_long_entry': 15.0, 'rsi_short_entry': 85.0},
    {'momentum_window': 25, 'rsi_long_entry': 10.0, 'rsi_short_entry': 90.0},
    {'momentum_window': 20, 'hold_days': 5, 'rsi_long_entry': 10.0, 'rsi_short_entry': 90.0},
]

wf_results = walk_forward_test(
    close, open_, high, low, nifty_close,
    param_grid, train_months=12, test_months=3,
)

if not wf_results.empty:
    wf_results.to_csv(OUTPUT_DIR / 'walk_forward_results.csv', index=False)
    print(wf_results[['window_start', 'train_end', 'test_end',
                       'train_sharpe', 'oos_sharpe']].to_string(index=False))
    print(f"\nMean OOS Sharpe : {wf_results['oos_sharpe'].mean():.3f}")
    print(f"% Positive OOS  : {(wf_results['oos_sharpe'] > 0).mean()*100:.1f}%")

window_start  train_end   test_end  train_sharpe  oos_sharpe
  2019-01-02 2020-01-02 2020-04-02       -3.7860     -2.4720
  2020-01-02 2021-01-02 2021-04-02        0.5360      1.9630
  2021-01-02 2022-01-02 2022-04-02       -0.2690     -0.1790
  2022-01-02 2023-01-02 2023-04-02       -0.3140     -3.8000
  2023-01-02 2024-01-02 2024-04-02       -1.2320     -0.6650

Mean OOS Sharpe : -1.031
% Positive OOS  : 20.0%


In [23]:
# Walk-forward Sharpe comparison
if not wf_results.empty:
    fig, ax = plt.subplots(figsize=(13, 5))
    x = range(len(wf_results))
    ax.bar([i - 0.2 for i in x], wf_results['train_sharpe'], 0.4,
           color='steelblue', alpha=0.8, label='In-sample Sharpe')
    ax.bar([i + 0.2 for i in x], wf_results['oos_sharpe'], 0.4,
           color='coral',     alpha=0.8, label='OOS Sharpe')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(
        [str(r['test_end']) for _, r in wf_results.iterrows()],
        rotation=30, ha='right', fontsize=8,
    )
    ax.set_title('Walk-Forward: In-Sample vs Out-of-Sample Sharpe', fontsize=13)
    ax.set_ylabel('Sharpe Ratio')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 6. Save Outputs

In [24]:
# Save trade log and summary
trade_log.to_csv(OUTPUT_DIR / 'trade_log.csv', index=False)
summary.to_csv(OUTPUT_DIR   / 'performance_summary.csv', index=False)
sens_df.to_csv(OUTPUT_DIR   / 'sensitivity_results.csv', index=False)

print('Saved:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:40s}  ({f.stat().st_size / 1024:.1f} KB)')

Saved:
  comparison_report.csv                     (0.3 KB)
  drawdown.png                              (112.7 KB)
  equity_curve.png                          (137.7 KB)
  monthly_heatmap.png                       (94.1 KB)
  performance_summary.csv                   (0.5 KB)
  sensitivity_results.csv                   (0.6 KB)
  stock_contribution.png                    (131.4 KB)
  trade_log.csv                             (51.7 KB)
  walk_forward_results.csv                  (0.7 KB)
